# Визуализация — мини-проект

Сквозной мини-проект на реальных данных. Повторяет ключевые темы урока.


In [ ]:
# Setup
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)


## Постановка задачи `(intro)`


EDA **Titanic** одним набором графиков: line/scatter/bar/hist/box, seaborn, heatmap, категориальные plot, subplots — все темы урока в одном сценарии.


## Загрузка данных `(eda)`


In [ ]:
warnings.filterwarnings("ignore")


sns.set_theme(style="whitegrid", font_scale=1.05)

raw = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
df["Survived"] = df["survived"].astype(int)
df["Pclass"] = df["pclass"].astype(int)
df["Age"] = pd.to_numeric(df["age"], errors="coerce")
df["Fare"] = pd.to_numeric(df["fare"], errors="coerce")
df["Sex"] = df["sex"]
print(df.shape)
df.head()


## Распределения: hist и box `(viz)`


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["Age"].dropna(), bins=30, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Age")
axes[0].set_title("Histogram возраста")
sns.boxplot(data=df, x="Pclass", y="Fare", hue="Pclass", ax=axes[1], palette="Blues", legend=False)
axes[1].set_title("Boxplot Fare по классу")
plt.tight_layout()
plt.show()


## Scatter и bar `(viz)`


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for survived, color, label in [(0, "crimson", "не выжил"), (1, "steelblue", "выжил")]:
    sub = df[df["Survived"] == survived]
    axes[0].scatter(sub["Age"], sub["Fare"], alpha=0.35, s=15, c=color, label=label)
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Fare")
axes[0].legend()
axes[0].set_title("Scatter Age vs Fare")

rates = df.groupby("Pclass")["Survived"].mean()
axes[1].bar(rates.index.astype(str), rates.values, color="steelblue")
axes[1].set_ylim(0, 1)
axes[1].set_xlabel("Pclass")
axes[1].set_ylabel("доля выживших")
axes[1].set_title("Bar: survival by class")
plt.tight_layout()
plt.show()


## Heatmap корреляций `(viz)`


In [ ]:
num = df[["Survived", "Pclass", "Age", "Fare"]].dropna()
corr = num.corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Корреляции числовых признаков")
plt.tight_layout()
plt.show()


## Категориальные графики seaborn `(viz)`


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x="Sex", hue="Sex", ax=axes[0], palette="Set2", legend=False)
sns.barplot(data=df, x="Sex", y="Survived", hue="Sex", estimator="mean", ax=axes[1], palette="muted", legend=False)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("доля выживших")
fig.suptitle("Категориальные plot")
plt.tight_layout()
plt.show()


## Subplots 2×2 — сводная EDA `(viz)`


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].hist(df["Fare"].dropna(), bins=40, color="steelblue", edgecolor="white")
axes[0, 0].set_title("hist Fare")
axes[0, 1].scatter(df["Age"], df["Fare"], c=df["Survived"], alpha=0.3, s=10, cmap="coolwarm")
axes[0, 1].set_title("scatter c=Survived")
df["Survived"].value_counts().plot(kind="bar", ax=axes[1, 0], color=["crimson", "steelblue"])
axes[1, 0].set_title("bar Survived")
sns.boxplot(data=df, x="Sex", y="Age", hue="Sex", ax=axes[1, 1], palette="pastel", legend=False)
axes[1, 1].set_title("box Age by Sex")
plt.tight_layout()
plt.show()


## ML-диагностика: остатки и ROC `(viz)`


In [ ]:
X = df[["Pclass", "Age", "Fare"]]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42),
)
pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
residual_proxy = y_test.values - proba
axes[0].scatter(proba, residual_proxy, alpha=0.5)
axes[0].axhline(0, color="crimson", ls="--")
axes[0].set_xlabel("P(survive)")
axes[0].set_ylabel("y - proba")
axes[0].set_title("Диагностика вероятностей")
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1])
plt.tight_layout()
plt.show()


## Чек-лист мини-проекта `(summary)`


1. Один датасет — все типы графиков.
2. Bar chart — ось Y с нуля.
3. Heatmap для corr; countplot/barplot для категорий.
4. subplots 2×2 для компактной EDA.
5. После модели — residual-style plot и ROC.
6. Подписи осей на русском/английском последовательно.
